# **FRAMEWORK V7 DATASET MAESTRO - MODELADO (C14 AJUSTADO)**

Versión ajustada para Exp01 y Exp04. Para regresión, además del escalamiento de X con entrenamiento, se escala el objetivo exclusivamente con `y_train`, se entrena con pérdida Huber y las predicciones se devuelven a la escala original antes de evaluar. Incluye baseline temporal de referencia.


## **M0. Configuración del Experimento**


In [ ]:
#==========================================================================================
# M0.1 CONFIGURACIÓN GENERAL
#==========================================================================================

import numpy as np

# Cambiar únicamente esta variable para ejecutar el experimento base.
EXPERIMENTO = "Exp04"   # Opciones actuales: "Exp01", "Exp04"

# Variante opcional de modelado. Ejemplos: "", "-V2", "-V3".
# La fuente de C13 sigue siendo EXPERIMENTO; la variante solo cambia la salida de C14.
VARIANTE_EXPERIMENTO = ""

ID_EJECUCION = f"{EXPERIMENTO}{VARIANTE_EXPERIMENTO}"

# Reproducibilidad
SEED = 42

# División temporal por Nodo
PORCENTAJE_TRAIN = 0.70
PORCENTAJE_VALIDACION = 0.15
PORCENTAJE_PRUEBA = 0.15

# Balanceo. Solo aplica a clasificación.
USAR_CLASS_WEIGHT = False
USAR_SMOTE = False

# Umbral para clasificación binaria
UMBRAL_CLASIFICACION = 0.50

# Validación de porcentajes
if not np.isclose(
    PORCENTAJE_TRAIN + PORCENTAJE_VALIDACION + PORCENTAJE_PRUEBA,
    1.0
):
    raise ValueError("Los porcentajes de train/validación/prueba deben sumar 1.0.")

if USAR_CLASS_WEIGHT and USAR_SMOTE:
    raise ValueError("No active USAR_CLASS_WEIGHT y USAR_SMOTE simultáneamente.")

print()
print("=" * 90)
print("CONFIGURACIÓN DEL EXPERIMENTO")
print("=" * 90)
print()
print(f"Experimento fuente C13 : {EXPERIMENTO}")
print(f"ID de ejecución C14    : {ID_EJECUCION}")
print(f"Train / Valid / Test   : "
      f"{PORCENTAJE_TRAIN:.0%} / "
      f"{PORCENTAJE_VALIDACION:.0%} / "
      f"{PORCENTAJE_PRUEBA:.0%}")
print(f"Class Weight           : {USAR_CLASS_WEIGHT}")
print(f"SMOTE                  : {USAR_SMOTE}")
print(f"Semilla                 : {SEED}")


In [ ]:
#==========================================================================================
# M0.2 LIBRERÍAS Y DIRECTORIOS
#==========================================================================================

import os
import re
import json
import math
import hashlib
import unicodedata
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from sklearn.preprocessing import (
    RobustScaler,
    StandardScaler,
    MinMaxScaler
)

tf.keras.utils.set_random_seed(SEED)

CARPETA_TENSORES = f"tensores/{ID_EJECUCION}"
CARPETA_MODELOS = f"modelos/{ID_EJECUCION}"
CARPETA_DIAGNOSTICOS = f"diagnosticos/{ID_EJECUCION}"
CARPETA_RESULTADOS = f"resultados/{ID_EJECUCION}"

for carpeta in [
    CARPETA_TENSORES,
    CARPETA_MODELOS,
    CARPETA_DIAGNOSTICOS,
    CARPETA_RESULTADOS
]:
    os.makedirs(carpeta, exist_ok=True)

print()
print("=" * 90)
print("LIBRERÍAS Y DIRECTORIOS")
print("=" * 90)
print()
print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print()
print("Directorios preparados:")
print("-", CARPETA_TENSORES)
print("-", CARPETA_MODELOS)
print("-", CARPETA_DIAGNOSTICOS)
print("-", CARPETA_RESULTADOS)


## **M1. Carga y Validación de Artefactos de C13**


In [ ]:
#==========================================================================================
# M1.1 DESCARGA AUTOMÁTICA DE ARTEFACTOS DE C13
#==========================================================================================

BASE_C13 = (
    "https://raw.githubusercontent.com/jriatiga/FRAMEWORK_V7/"
    "refs/heads/main/DATA/MACHINE_LEARNING/C13_MACHINE_LEARNING/"
    f"Transformaciones/{EXPERIMENTO}/"
)

ARCHIVOS_C13 = [
    "secuencias_X.npy",
    "secuencias_y.npy",
    "metadata_secuencias.csv",
    "metadata_secuencias_detalle.csv",
    "registro_preparacion.csv"
]

def descargar_archivo(url, ruta_local):
    respuesta = requests.get(url, timeout=120)
    respuesta.raise_for_status()

    with open(ruta_local, "wb") as archivo:
        archivo.write(respuesta.content)

    if os.path.getsize(ruta_local) == 0:
        raise IOError(f"El archivo descargado está vacío: {ruta_local}")

    return ruta_local

for nombre in ARCHIVOS_C13:
    url = BASE_C13 + nombre
    descargar_archivo(url, nombre)

print()
print("=" * 90)
print("ARTEFACTOS C13 DESCARGADOS")
print("=" * 90)
print()

for nombre in ARCHIVOS_C13:
    print(f"{nombre:<40} : {os.path.getsize(nombre):>10} bytes")


In [ ]:
#==========================================================================================
# M1.2 CARGA Y RECUPERACIÓN DE CONFIGURACIÓN
#==========================================================================================

secuencias_X_raw = np.load(
    "secuencias_X.npy",
    allow_pickle=False
)

secuencias_y_raw = np.load(
    "secuencias_y.npy",
    allow_pickle=False
)

metadata_secuencias = pd.read_csv(
    "metadata_secuencias.csv",
    encoding="utf-8-sig"
)

metadata_detalle = pd.read_csv(
    "metadata_secuencias_detalle.csv",
    encoding="utf-8-sig"
)

registro_preparacion = pd.read_csv(
    "registro_preparacion.csv",
    encoding="utf-8-sig"
)

def normalizar_parametro(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(
        caracter for caracter in texto
        if not unicodedata.combining(caracter)
    )
    texto = texto.lower().strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto

def obtener_parametro(df, nombres, default=None, requerido=True):
    if isinstance(nombres, str):
        nombres = [nombres]

    mapa = {
        normalizar_parametro(parametro): valor
        for parametro, valor in zip(df["Parametro"], df["Valor"])
    }

    for nombre in nombres:
        clave = normalizar_parametro(nombre)
        if clave in mapa and pd.notna(mapa[clave]):
            return mapa[clave]

    if requerido:
        raise KeyError(
            "No se encontró ninguno de estos parámetros: "
            + ", ".join(nombres)
        )

    return default

def convertir_bool(valor):
    return normalizar_parametro(valor) in {
        "true", "1", "si", "yes", "verdadero"
    }

DOMINIO = str(
    obtener_parametro(
        registro_preparacion,
        "Dominio"
    )
)

TIPO_PROBLEMA = str(
    obtener_parametro(
        metadata_secuencias,
        ["Tipo Problema", "Tipo de Problema"],
        default=obtener_parametro(
            registro_preparacion,
            ["Tipo de Problema", "Tipo Problema"]
        ),
        requerido=False
    )
)

TIPO_MODELO = str(
    obtener_parametro(
        metadata_secuencias,
        "Modelo",
        default=obtener_parametro(
            registro_preparacion,
            "Modelo",
            default="LSTM",
            requerido=False
        ),
        requerido=False
    )
)

VARIABLE_OBJETIVO = str(
    obtener_parametro(
        registro_preparacion,
        "Variable Objetivo"
    )
)

VARIABLE_OBJETIVO_CIENTIFICO = str(
    obtener_parametro(
        registro_preparacion,
        [
            "Variable Objetivo Cientifico",
            "Variable Objetivo Científico"
        ],
        default=VARIABLE_OBJETIVO,
        requerido=False
    )
)

VENTANA = int(float(
    obtener_parametro(
        registro_preparacion,
        ["Ventana Temporal", "Ventana"]
    )
))

HORIZONTE = int(float(
    obtener_parametro(
        registro_preparacion,
        "Horizonte"
    )
))

METODO_TRANSFORMACION = str(
    obtener_parametro(
        registro_preparacion,
        ["Método de Transformación", "Metodo de Transformacion"]
    )
)

TRANSFORMACION_C13 = convertir_bool(
    obtener_parametro(
        registro_preparacion,
        ["Transformación Aplicada en C13", "Transformacion Aplicada en C13"],
        default=False,
        requerido=False
    )
)

NUM_SECUENCIAS_REGISTRADAS = int(float(
    obtener_parametro(
        registro_preparacion,
        ["Número de Secuencias", "Numero de Secuencias"]
    )
))

LISTA_VARIABLES = obtener_parametro(
    registro_preparacion,
    ["Lista Variables Predictoras"],
    default=None,
    requerido=False
)

if LISTA_VARIABLES is None:
    LISTA_VARIABLES = obtener_parametro(
        metadata_secuencias,
        "Variables Predictoras"
    )

VARIABLES_MODELO = [
    variable.strip()
    for variable in str(LISTA_VARIABLES).split(";")
    if variable.strip()
]

print()
print("=" * 90)
print("CONFIGURACIÓN RECUPERADA DE C13")
print("=" * 90)
print()
print(f"Experimento                 : {EXPERIMENTO}")
print(f"Dominio                     : {DOMINIO}")
print(f"Tipo de problema            : {TIPO_PROBLEMA}")
print(f"Modelo                      : {TIPO_MODELO}")
print(f"Objetivo del modelo         : {VARIABLE_OBJETIVO}")
print(f"Objetivo científico         : {VARIABLE_OBJETIVO_CIENTIFICO}")
print(f"Ventana                     : {VENTANA}")
print(f"Horizonte                   : {HORIZONTE}")
print(f"Variables predictoras       : {len(VARIABLES_MODELO)}")
print(f"Método transformación       : {METODO_TRANSFORMACION}")
print(f"Transformación aplicada C13 : {TRANSFORMACION_C13}")
print(f"Secuencias registradas      : {NUM_SECUENCIAS_REGISTRADAS}")


In [ ]:
#==========================================================================================
# M1.3 VALIDACIÓN INTEGRAL DE LOS ARTEFACTOS DE ENTRADA
#==========================================================================================

metadata_detalle["Fecha_Inicio"] = pd.to_datetime(
    metadata_detalle["Fecha_Inicio"],
    errors="raise"
)

metadata_detalle["Fecha_Fin"] = pd.to_datetime(
    metadata_detalle["Fecha_Fin"],
    errors="raise"
)

metadata_detalle["Fecha_Objetivo"] = pd.to_datetime(
    metadata_detalle["Fecha_Objetivo"],
    errors="raise"
)

if "Indice_Secuencia" not in metadata_detalle.columns:
    raise ValueError(
        "metadata_secuencias_detalle.csv no contiene 'Indice_Secuencia'. "
        "Use los artefactos actualizados de C13."
    )

metadata_detalle["Indice_Secuencia"] = (
    metadata_detalle["Indice_Secuencia"].astype(int)
)

controles_entrada = {
    "X_es_tensor_3D":
        secuencias_X_raw.ndim == 3,

    "y_es_vector_1D":
        secuencias_y_raw.ndim == 1,

    "Alineacion_X_y":
        len(secuencias_X_raw) == len(secuencias_y_raw),

    "Alineacion_X_metadata":
        len(secuencias_X_raw) == len(metadata_detalle),

    "Secuencias_coinciden_registro":
        len(secuencias_X_raw) == NUM_SECUENCIAS_REGISTRADAS,

    "Ventana_coincide":
        secuencias_X_raw.shape[1] == VENTANA,

    "Predictoras_coinciden":
        secuencias_X_raw.shape[2] == len(VARIABLES_MODELO),

    "Indices_secuenciales":
        np.array_equal(
            metadata_detalle["Indice_Secuencia"].to_numpy(),
            np.arange(len(metadata_detalle))
        ),

    "Sin_NaN_en_X":
        not pd.isna(secuencias_X_raw).any(),

    "Sin_NaN_en_y":
        not pd.isna(secuencias_y_raw).any(),

    "Fechas_temporales_validas":
        metadata_detalle[
            ["Fecha_Inicio", "Fecha_Fin", "Fecha_Objetivo"]
        ].notna().all().all(),

    "Orden_ventana_objetivo":
        (
            metadata_detalle["Fecha_Inicio"]
            <= metadata_detalle["Fecha_Fin"]
        ).all()
        and
        (
            metadata_detalle["Fecha_Fin"]
            < metadata_detalle["Fecha_Objetivo"]
        ).all(),

    "Sin_duplicados_Nodo_FechaObjetivo":
        not metadata_detalle.duplicated(
            subset=["Nodo", "Fecha_Objetivo"]
        ).any(),

    "C13_sin_escalamiento":
        not TRANSFORMACION_C13
}

if TIPO_PROBLEMA.lower().startswith("clas"):
    controles_entrada["Clasificacion_con_2_clases"] = (
        len(np.unique(secuencias_y_raw)) >= 2
    )

if not all(controles_entrada.values()):
    print()
    print("=" * 90)
    print("VALIDACIÓN DE ENTRADA")
    print("=" * 90)
    print()

    for control, estado_control in controles_entrada.items():
        print(f"- {control:<40}: {estado_control}")

    raise ValueError(
        "Los artefactos de C13 no superaron la validación de C14. "
        "Revise especialmente si GitHub contiene una versión antigua ya escalada."
    )

print()
print("=" * 90)
print("VALIDACIÓN DE ENTRADA")
print("=" * 90)
print()

for control, estado_control in controles_entrada.items():
    print(f"- {control:<40}: {estado_control}")

print()
print("Resultado : ARTEFACTOS C13 VÁLIDOS")
print()
print(f"Shape X raw : {secuencias_X_raw.shape}")
print(f"Shape y     : {secuencias_y_raw.shape}")
print(f"Nodos       : {metadata_detalle['Nodo'].nunique()}")


## **M2. División Temporal por Nodo**

La división se hace sobre `Fecha_Objetivo` dentro de cada Nodo. No se mezclan secuencias de un Nodo con otro para determinar los cortes y no se usa información del conjunto de prueba para ajustar transformaciones.


In [ ]:
#==========================================================================================
# M2.1 CONSTRUCCIÓN DEL SPLIT TEMPORAL POR NODO
#==========================================================================================

metadata_split = metadata_detalle.copy()

metadata_split["Particion"] = None

resumen_split_nodos = []

for nodo, grupo in metadata_split.groupby("Nodo", sort=True):

    grupo = grupo.sort_values(
        "Fecha_Objetivo"
    )

    n_nodo = len(grupo)

    corte_train = int(
        np.floor(
            n_nodo * PORCENTAJE_TRAIN
        )
    )

    corte_valid = int(
        np.floor(
            n_nodo * (
                PORCENTAJE_TRAIN
                + PORCENTAJE_VALIDACION
            )
        )
    )

    if corte_train < 1 or corte_valid <= corte_train or corte_valid >= n_nodo:
        raise ValueError(
            f"El Nodo '{nodo}' no tiene suficientes secuencias "
            "para crear train/validación/prueba."
        )

    indices_ordenados = grupo["Indice_Secuencia"].to_numpy()

    idx_nodo_train = indices_ordenados[:corte_train]
    idx_nodo_valid = indices_ordenados[corte_train:corte_valid]
    idx_nodo_test = indices_ordenados[corte_valid:]

    metadata_split.loc[
        metadata_split["Indice_Secuencia"].isin(idx_nodo_train),
        "Particion"
    ] = "Train"

    metadata_split.loc[
        metadata_split["Indice_Secuencia"].isin(idx_nodo_valid),
        "Particion"
    ] = "Validacion"

    metadata_split.loc[
        metadata_split["Indice_Secuencia"].isin(idx_nodo_test),
        "Particion"
    ] = "Prueba"

    resumen_split_nodos.append({
        "Nodo": nodo,
        "Total": n_nodo,
        "Train": len(idx_nodo_train),
        "Validacion": len(idx_nodo_valid),
        "Prueba": len(idx_nodo_test),
        "Ultima_Fecha_Train": grupo.iloc[corte_train - 1]["Fecha_Objetivo"],
        "Primera_Fecha_Validacion": grupo.iloc[corte_train]["Fecha_Objetivo"],
        "Ultima_Fecha_Validacion": grupo.iloc[corte_valid - 1]["Fecha_Objetivo"],
        "Primera_Fecha_Prueba": grupo.iloc[corte_valid]["Fecha_Objetivo"]
    })

resumen_split_nodos = pd.DataFrame(
    resumen_split_nodos
)

if metadata_split["Particion"].isna().any():
    raise ValueError("Existen secuencias sin partición temporal.")

# Orden global determinista por fecha y Nodo dentro de cada partición
def indices_particion(nombre):
    return (
        metadata_split.loc[
            metadata_split["Particion"] == nombre
        ]
        .sort_values(
            ["Fecha_Objetivo", "Nodo", "Indice_Secuencia"]
        )["Indice_Secuencia"]
        .to_numpy(dtype=int)
    )

idx_train = indices_particion("Train")
idx_valid = indices_particion("Validacion")
idx_test = indices_particion("Prueba")

# Comprobar que cada secuencia pertenece a una sola partición
indices_totales = np.concatenate(
    [idx_train, idx_valid, idx_test]
)

if (
    len(indices_totales) != len(secuencias_X_raw)
    or len(np.unique(indices_totales)) != len(secuencias_X_raw)
):
    raise ValueError("El split contiene duplicados o secuencias sin asignar.")

print()
print("=" * 90)
print("DIVISIÓN TEMPORAL POR NODO")
print("=" * 90)
print()
display(resumen_split_nodos)


In [ ]:
#==========================================================================================
# M2.2 VALIDACIÓN TEMPORAL Y CONSTRUCCIÓN DE PARTICIONES RAW
#==========================================================================================

# Verificar estrictamente los cortes cronológicos dentro de cada Nodo.
integridad_temporal_split = True

for _, fila in resumen_split_nodos.iterrows():

    if not (
        fila["Ultima_Fecha_Train"]
        < fila["Primera_Fecha_Validacion"]
        <= fila["Ultima_Fecha_Validacion"]
        < fila["Primera_Fecha_Prueba"]
    ):
        integridad_temporal_split = False
        break

if not integridad_temporal_split:
    raise ValueError("Se detectó una inconsistencia temporal en el split.")

X_raw = np.asarray(
    secuencias_X_raw,
    dtype=np.float32
)

if TIPO_PROBLEMA.lower().startswith("clas"):
    y = np.asarray(
        secuencias_y_raw,
        dtype=np.int32
    )
else:
    y = np.asarray(
        secuencias_y_raw,
        dtype=np.float32
    )

X_train_raw = X_raw[idx_train]
y_train = y[idx_train]

X_valid_raw = X_raw[idx_valid]
y_valid = y[idx_valid]

X_test_raw = X_raw[idx_test]
y_test = y[idx_test]

print()
print("=" * 90)
print("RESUMEN DEL SPLIT")
print("=" * 90)
print()
print(f"Total         : {len(X_raw)}")
print(f"Entrenamiento : {X_train_raw.shape}")
print(f"Validación    : {X_valid_raw.shape}")
print(f"Prueba        : {X_test_raw.shape}")
print(f"Integridad temporal : {integridad_temporal_split}")

if TIPO_PROBLEMA.lower().startswith("clas"):

    print()
    print("Distribución de clases por partición:")

    for nombre, vector in [
        ("Train", y_train),
        ("Validación", y_valid),
        ("Prueba", y_test)
    ]:
        valores, conteos = np.unique(
            vector,
            return_counts=True
        )

        distribucion = dict(
            zip(
                valores.astype(int).tolist(),
                conteos.tolist()
            )
        )

        print(f"- {nombre:<10}: {distribucion}")

    if len(np.unique(y_valid)) < 2:
        print()
        print("ADVERTENCIA: validación temporal contiene una sola clase.")

    if len(np.unique(y_test)) < 2:
        print("ADVERTENCIA: prueba temporal contiene una sola clase.")

else:

    print()
    print("Rango del objetivo por partición:")
    print(f"- Train      : {y_train.min():.4f} -> {y_train.max():.4f}")
    print(f"- Validación : {y_valid.min():.4f} -> {y_valid.max():.4f}")
    print(f"- Prueba     : {y_test.min():.4f} -> {y_test.max():.4f}")


## **M3. Escalamiento sin Fuga de Información**


In [ ]:
#==========================================================================================
# M3.1 ESCALAMIENTO SIN FUGA: X Y, SOLO PARA REGRESIÓN, y
#==========================================================================================

def crear_scaler(nombre_metodo):
    metodo = normalizar_parametro(nombre_metodo).replace(" ", "")
    if "robustscaler" in metodo or metodo == "robust":
        return RobustScaler()
    if "standardscaler" in metodo or metodo == "standard":
        return StandardScaler()
    if "minmaxscaler" in metodo or metodo == "minmax":
        return MinMaxScaler()
    if metodo in {"none", "ninguno", "sintransformacion", "noaplica"}:
        return None
    raise ValueError(f"Método de transformación no soportado: {nombre_metodo}")

scaler = crear_scaler(METODO_TRANSFORMACION)
NUM_VARIABLES = X_train_raw.shape[2]

def a_2d(tensor):
    return tensor.reshape(-1, tensor.shape[2])

def transformar_3d(tensor, transformador):
    if transformador is None:
        return tensor.astype(np.float32)
    forma = tensor.shape
    transformado = transformador.transform(a_2d(tensor))
    return transformado.reshape(forma).astype(np.float32)

#------------------------------------------------------------------------------------------
# Escalamiento de X: fit SOLO con X_train
#------------------------------------------------------------------------------------------
if scaler is not None:
    scaler.fit(a_2d(X_train_raw))
    X_train = transformar_3d(X_train_raw, scaler)
    X_valid = transformar_3d(X_valid_raw, scaler)
    X_test = transformar_3d(X_test_raw, scaler)
    X = transformar_3d(X_raw, scaler)
    RUTA_SCALER = os.path.join(CARPETA_MODELOS, "scaler.pkl")
    joblib.dump(scaler, RUTA_SCALER)
else:
    X_train = X_train_raw.copy()
    X_valid = X_valid_raw.copy()
    X_test = X_test_raw.copy()
    X = X_raw.copy()
    RUTA_SCALER = None

#------------------------------------------------------------------------------------------
# Escalamiento del objetivo SOLO para regresión: fit SOLO con y_train
# La evaluación y los archivos tensor_y.npy permanecen en la escala original.
#------------------------------------------------------------------------------------------
scaler_y = None
RUTA_SCALER_Y = None
TRANSFORMACION_OBJETIVO = "Ninguna"

if TIPO_PROBLEMA.lower().startswith("reg"):
    scaler_y = StandardScaler()
    scaler_y.fit(y_train.reshape(-1, 1))

    y_train_modelo = scaler_y.transform(y_train.reshape(-1, 1)).reshape(-1).astype(np.float32)
    y_valid_modelo = scaler_y.transform(y_valid.reshape(-1, 1)).reshape(-1).astype(np.float32)
    y_test_modelo = scaler_y.transform(y_test.reshape(-1, 1)).reshape(-1).astype(np.float32)

    RUTA_SCALER_Y = os.path.join(CARPETA_MODELOS, "scaler_y.pkl")
    joblib.dump(scaler_y, RUTA_SCALER_Y)
    TRANSFORMACION_OBJETIVO = "StandardScaler"
else:
    y_train_modelo = y_train.copy()
    y_valid_modelo = y_valid.copy()
    y_test_modelo = y_test.copy()

print()
print("=" * 90)
print("ESCALAMIENTO SIN FUGA")
print("=" * 90)
print()
print(f"Método X                : {METODO_TRANSFORMACION}")
print("Ajuste scaler X         : SOLO X_train")
print(f"Shape X_train           : {X_train.shape}")
print(f"Shape X_valid           : {X_valid.shape}")
print(f"Shape X_test            : {X_test.shape}")
print(f"NaN después de escalar X: {pd.isna(X).sum()}")
print(f"Scaler X                : {RUTA_SCALER}")
print()
print(f"Transformación objetivo : {TRANSFORMACION_OBJETIVO}")
print("Ajuste scaler y         : SOLO y_train" if RUTA_SCALER_Y else "Ajuste scaler y         : No aplica")
print(f"Scaler y                : {RUTA_SCALER_Y}")

if pd.isna(X).any():
    raise ValueError("El escalamiento de X produjo valores NaN.")

if pd.isna(y_train_modelo).any() or pd.isna(y_valid_modelo).any() or pd.isna(y_test_modelo).any():
    raise ValueError("La transformación del objetivo produjo valores NaN.")


## **M4. Tensores y Metadata de Modelado**


In [ ]:
#==========================================================================================
# M4.1 EXPORTACIÓN DE TENSORES, SPLIT Y METADATA
#==========================================================================================

RUTA_TENSOR_X = os.path.join(
    CARPETA_TENSORES,
    "tensor_X.npy"
)

RUTA_TENSOR_X_RAW = os.path.join(
    CARPETA_TENSORES,
    "tensor_X_raw.npy"
)

RUTA_TENSOR_Y = os.path.join(
    CARPETA_TENSORES,
    "tensor_y.npy"
)

RUTA_METADATA_SPLIT = os.path.join(
    CARPETA_TENSORES,
    "metadata_split.csv"
)

np.save(
    RUTA_TENSOR_X,
    X
)

np.save(
    RUTA_TENSOR_X_RAW,
    X_raw
)

np.save(
    RUTA_TENSOR_Y,
    y
)

metadata_split.to_csv(
    RUTA_METADATA_SPLIT,
    index=False,
    encoding="utf-8-sig"
)

metadata_tensor = pd.DataFrame({
    "Parametro": [
        "Experimento",
        "Experimento Fuente C13",
        "Dominio",
        "Tipo Problema",
        "Modelo",
        "Variable Objetivo",
        "Variable Objetivo Cientifico",
        "Ventana",
        "Horizonte",
        "Variables Predictoras",
        "Variables Modelo",
        "Metodo Transformacion",
        "Transformacion Aplicada en C13",
        "Scaler Ajustado En",
        "Scaler",
        "Transformacion Objetivo",
        "Scaler Objetivo Ajustado En",
        "Scaler Objetivo",
        "Numero Secuencias",
        "Muestras Train",
        "Muestras Validacion",
        "Muestras Prueba",
        "Estrategia Split",
        "Fecha Generacion",
        "Tensor X",
        "Tensor X Raw",
        "Tensor y",
        "Metadata Split"
    ],
    "Valor": [
        ID_EJECUCION,
        EXPERIMENTO,
        DOMINIO,
        TIPO_PROBLEMA,
        TIPO_MODELO,
        VARIABLE_OBJETIVO,
        VARIABLE_OBJETIVO_CIENTIFICO,
        VENTANA,
        HORIZONTE,
        X.shape[2],
        ";".join(VARIABLES_MODELO),
        METODO_TRANSFORMACION,
        False,
        "C14 - exclusivamente conjunto de entrenamiento",
        "scaler.pkl" if RUTA_SCALER else "",
        TRANSFORMACION_OBJETIVO,
        "C14 - exclusivamente y_train" if RUTA_SCALER_Y else "No aplica",
        "scaler_y.pkl" if RUTA_SCALER_Y else "",
        X.shape[0],
        len(X_train),
        len(X_valid),
        len(X_test),
        "Temporal por Nodo usando Fecha_Objetivo",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "tensor_X.npy",
        "tensor_X_raw.npy",
        "tensor_y.npy",
        "metadata_split.csv"
    ]
})

RUTA_METADATA_TENSOR_CSV = os.path.join(
    CARPETA_TENSORES,
    "metadata_tensor.csv"
)

RUTA_METADATA_TENSOR_XLSX = os.path.join(
    CARPETA_TENSORES,
    "metadata_tensor.xlsx"
)

metadata_tensor.to_csv(
    RUTA_METADATA_TENSOR_CSV,
    index=False,
    encoding="utf-8-sig"
)

metadata_tensor.to_excel(
    RUTA_METADATA_TENSOR_XLSX,
    index=False
)

print()
print("=" * 90)
print("TENSORES Y METADATA EXPORTADOS")
print("=" * 90)
print()
print(RUTA_TENSOR_X)
print(RUTA_TENSOR_X_RAW)
print(RUTA_TENSOR_Y)
print(RUTA_METADATA_SPLIT)
print(RUTA_METADATA_TENSOR_CSV)
print(RUTA_METADATA_TENSOR_XLSX)
print()
display(metadata_tensor)


## **M5. Construcción y Entrenamiento del Modelo**


In [ ]:
#==========================================================================================
# M5.1 CONFIGURACIÓN DEL MODELO
#==========================================================================================

UNIDADES_LSTM = 32
DROPOUT = 0.20
ACTIVACION_LSTM = "tanh"

EPOCAS = 100
BATCH_SIZE = 32
LEARNING_RATE = (
    0.0005
    if TIPO_PROBLEMA.lower().startswith("reg")
    else 0.001
)
PACIENCIA = 12

if TIPO_MODELO.upper() != "LSTM":
    raise ValueError(
        f"Modelo no soportado en esta versión de C14: {TIPO_MODELO}"
    )

print()
print("=" * 90)
print("CONFIGURACIÓN DEL MODELO")
print("=" * 90)
print()
print(f"Modelo             : {TIPO_MODELO}")
print(f"Unidades LSTM      : {UNIDADES_LSTM}")
print(f"Dropout            : {DROPOUT}")
print(f"Activación LSTM    : {ACTIVACION_LSTM}")
print(f"Épocas máximas     : {EPOCAS}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Learning rate      : {LEARNING_RATE}")
print(f"Early stopping     : {PACIENCIA}")


In [ ]:
#==========================================================================================
# M5.2 CONSTRUCCIÓN Y COMPILACIÓN
#==========================================================================================

capas = [
    keras.layers.Input(
        shape=(X_train.shape[1], X_train.shape[2])
    ),
    keras.layers.LSTM(
        units=UNIDADES_LSTM,
        activation=ACTIVACION_LSTM
    ),
    keras.layers.Dropout(DROPOUT)
]

if TIPO_PROBLEMA.lower().startswith("clas"):
    capas.append(
        keras.layers.Dense(1, activation="sigmoid")
    )
    LOSS = "binary_crossentropy"
    METRICAS = [
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]

elif TIPO_PROBLEMA.lower().startswith("reg"):
    # y se entrena estandarizada con parámetros calculados SOLO sobre y_train.
    # Una capa densa pequeña aporta capacidad no lineal sin inflar demasiado el modelo.
    capas.extend([
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dropout(0.10),
        keras.layers.Dense(1, activation="linear")
    ])
    LOSS = keras.losses.Huber(delta=1.0)
    METRICAS = [
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.RootMeanSquaredError(name="rmse")
    ]
else:
    raise ValueError(f"Tipo de problema no soportado: {TIPO_PROBLEMA}")

modelo = keras.Sequential(capas, name=f"LSTM_{ID_EJECUCION}")
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=LOSS,
    metrics=METRICAS
)

print()
print("=" * 90)
print("MODELO COMPILADO")
print("=" * 90)
print()
print(f"Tipo problema : {TIPO_PROBLEMA}")
print(f"Loss          : {LOSS}")
print(f"Objetivo escalado para entrenamiento : {RUTA_SCALER_Y is not None}")
print()
modelo.summary()


In [ ]:
#==========================================================================================
# M5.3 BALANCEO DEL CONJUNTO DE ENTRENAMIENTO
#==========================================================================================

from sklearn.utils.class_weight import compute_class_weight

class_weight = None
X_entrenamiento = X_train
y_entrenamiento = y_train_modelo

if TIPO_PROBLEMA.lower().startswith("reg"):

    if USAR_CLASS_WEIGHT or USAR_SMOTE:
        print(
            "ADVERTENCIA: balanceo desactivado porque Exp04 es un problema de regresión."
        )

elif TIPO_PROBLEMA.lower().startswith("clas"):

    if USAR_CLASS_WEIGHT:

        clases_train = np.unique(
            y_train
        )

        pesos = compute_class_weight(
            class_weight="balanced",
            classes=clases_train,
            y=y_train
        )

        class_weight = {
            int(clase): float(peso)
            for clase, peso in zip(
                clases_train,
                pesos
            )
        }

    if USAR_SMOTE:

        from imblearn.over_sampling import SMOTE

        clases_train, conteos_train = np.unique(
            y_train,
            return_counts=True
        )

        minimo_clase = int(
            conteos_train.min()
        )

        if minimo_clase < 2:
            raise ValueError(
                "SMOTE no puede aplicarse: una clase tiene menos de 2 muestras."
            )

        k_neighbors = min(
            5,
            minimo_clase - 1
        )

        smote = SMOTE(
            random_state=SEED,
            k_neighbors=k_neighbors
        )

        X_train_2d = X_train.reshape(
            X_train.shape[0],
            -1
        )

        X_smote_2d, y_smote = smote.fit_resample(
            X_train_2d,
            y_train
        )

        X_entrenamiento = X_smote_2d.reshape(
            -1,
            X_train.shape[1],
            X_train.shape[2]
        ).astype(np.float32)

        y_entrenamiento = y_smote.astype(
            np.int32
        )

print()
print("=" * 90)
print("BALANCEO")
print("=" * 90)
print()
print(f"Class Weight : {class_weight}")
print(f"SMOTE        : {USAR_SMOTE}")
print(f"X usado      : {X_entrenamiento.shape}")
print(f"y usado      : {y_entrenamiento.shape}")


In [ ]:
#==========================================================================================
# M5.4 ENTRENAMIENTO
#==========================================================================================

import time

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PACIENCIA,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(3, PACIENCIA // 3),
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.TerminateOnNaN()
]

inicio = time.time()

historial = modelo.fit(
    X_entrenamiento,
    y_entrenamiento,
    validation_data=(
        X_valid,
        y_valid_modelo
    ),
    epochs=EPOCAS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
    class_weight=class_weight,
    shuffle=True
)

TIEMPO_ENTRENAMIENTO = (
    time.time() - inicio
)

historial_df = pd.DataFrame(
    historial.history
)

print()
print("=" * 90)
print("ENTRENAMIENTO FINALIZADO")
print("=" * 90)
print()
print(f"Épocas ejecutadas : {len(historial_df)}")
print(f"Tiempo (s)        : {TIEMPO_ENTRENAMIENTO:.2f}")
print()
display(historial_df.tail())


## **M6. Evaluación del Modelo**


In [ ]:
#==========================================================================================
# M6.1 EVALUACIÓN Y PREDICCIONES
#==========================================================================================

resultado_test = modelo.evaluate(
    X_test,
    y_test_modelo,
    verbose=0,
    return_dict=True
)

salida_modelo_interna = modelo.predict(
    X_test,
    verbose=0
).reshape(-1)

if TIPO_PROBLEMA.lower().startswith("reg"):
    # Volver a unidades originales antes de calcular métricas científicas.
    salida_modelo = scaler_y.inverse_transform(
        salida_modelo_interna.reshape(-1, 1)
    ).reshape(-1)
else:
    salida_modelo = salida_modelo_interna

metadata_test = (
    metadata_split.loc[
        metadata_split["Indice_Secuencia"].isin(idx_test),
        ["Indice_Secuencia", "Nodo", "Fecha_Inicio", "Fecha_Fin", "Fecha_Objetivo"]
    ]
    .set_index("Indice_Secuencia")
    .loc[idx_test]
    .reset_index()
)

print()
print("=" * 90)
print("EVALUACIÓN GENERAL")
print("=" * 90)
print()
for nombre_metrica, valor_metrica in resultado_test.items():
    print(f"{nombre_metrica:<20}: {valor_metrica:.6f}")

if TIPO_PROBLEMA.lower().startswith("reg"):
    print("Nota: loss/mae/rmse anteriores corresponden a la escala interna estandarizada de y.")
    print("Las métricas científicas de la siguiente celda se calculan en la escala original.")


In [ ]:
#==========================================================================================
# M6.2 MÉTRICAS ESPECÍFICAS SEGÚN EL TIPO DE PROBLEMA
#==========================================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

if TIPO_PROBLEMA.lower().startswith("clas"):

    probabilidades = salida_modelo

    predicciones = (
        probabilidades >= UMBRAL_CLASIFICACION
    ).astype(int)

    accuracy = accuracy_score(
        y_test,
        predicciones
    )

    precision = precision_score(
        y_test,
        predicciones,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predicciones,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predicciones,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        predicciones,
        labels=[0, 1]
    )

    TN, FP, FN, TP = cm.ravel()

    especificidad = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else np.nan
    )

    accuracy_balanceada = np.nanmean(
        [recall, especificidad]
    )

    if len(np.unique(y_test)) >= 2:
        auc_roc = roc_auc_score(
            y_test,
            probabilidades
        )
    else:
        auc_roc = np.nan

    comparacion = metadata_test.copy()
    comparacion["Real"] = y_test.astype(int)
    comparacion["Probabilidad"] = probabilidades
    comparacion["Prediccion"] = predicciones

    if len(np.unique(y_test)) < 2:
        estado = (
            "EVALUACION LIMITADA - TEST CON UNA SOLA CLASE"
        )
    elif (
        f1 >= 0.70
        and accuracy_balanceada >= 0.70
    ):
        estado = "MODELO APROBADO"
    elif (
        f1 >= 0.50
        and accuracy_balanceada >= 0.60
    ):
        estado = "MODELO ACEPTABLE"
    else:
        estado = "MODELO REQUIERE AJUSTES"

    print()
    print("=" * 90)
    print("MÉTRICAS DE CLASIFICACIÓN")
    print("=" * 90)
    print()
    print(f"Accuracy             : {accuracy:.4f}")
    print(f"Precision            : {precision:.4f}")
    print(f"Recall               : {recall:.4f}")
    print(f"F1-Score             : {f1:.4f}")
    print(f"Especificidad        : {especificidad:.4f}")
    print(f"Accuracy balanceada  : {accuracy_balanceada:.4f}")
    print(
        "AUC ROC              : "
        + (
            f"{auc_roc:.4f}"
            if np.isfinite(auc_roc)
            else "No calculable: test con una sola clase"
        )
    )
    print()
    print("Matriz de confusión [0, 1]:")
    print(cm)
    print()
    print(
        classification_report(
            y_test,
            predicciones,
            labels=[0, 1],
            digits=4,
            zero_division=0
        )
    )

else:

    predicciones = salida_modelo.astype(np.float64)
    y_test_eval = y_test.astype(np.float64)

    mae = mean_absolute_error(y_test_eval, predicciones)
    rmse = np.sqrt(mean_squared_error(y_test_eval, predicciones))
    r2 = r2_score(y_test_eval, predicciones)

    mascara_mape = np.abs(y_test_eval) > 1e-12
    if mascara_mape.any():
        mape = np.mean(
            np.abs(
                (y_test_eval[mascara_mape] - predicciones[mascara_mape])
                / y_test_eval[mascara_mape]
            )
        ) * 100
    else:
        mape = np.nan

    escala_media = np.mean(np.abs(y_test_eval))
    nmae = 100 * mae / escala_media if escala_media > 0 else np.nan
    nrmse = 100 * rmse / escala_media if escala_media > 0 else np.nan

    #--------------------------------------------------------------------------
    # Baseline 1: media del entrenamiento
    #--------------------------------------------------------------------------
    baseline_media = np.full_like(
        y_test_eval,
        fill_value=float(np.mean(y_train)),
        dtype=np.float64
    )
    mae_baseline_media = mean_absolute_error(y_test_eval, baseline_media)
    rmse_baseline_media = np.sqrt(mean_squared_error(y_test_eval, baseline_media))
    r2_baseline_media = r2_score(y_test_eval, baseline_media)

    #--------------------------------------------------------------------------
    # Baseline 2: persistencia estacional de 12 meses por Nodo, cuando existe
    #--------------------------------------------------------------------------
    metadata_y = metadata_detalle[["Indice_Secuencia", "Nodo", "Fecha_Objetivo"]].copy()
    metadata_y["Fecha_Objetivo"] = pd.to_datetime(metadata_y["Fecha_Objetivo"])
    metadata_y["y"] = y.astype(np.float64)

    mapa_y = {
        (fila.Nodo, fila.Fecha_Objetivo): float(fila.y)
        for fila in metadata_y.itertuples(index=False)
    }

    baseline_estacional = []
    for fila in metadata_test.itertuples(index=False):
        fecha_anterior = pd.Timestamp(fila.Fecha_Objetivo) - pd.DateOffset(years=1)
        baseline_estacional.append(
            mapa_y.get((fila.Nodo, fecha_anterior), np.nan)
        )

    baseline_estacional = np.asarray(baseline_estacional, dtype=np.float64)
    mascara_baseline = np.isfinite(baseline_estacional)

    if mascara_baseline.sum() >= 2:
        mae_baseline_estacional = mean_absolute_error(
            y_test_eval[mascara_baseline], baseline_estacional[mascara_baseline]
        )
        rmse_baseline_estacional = np.sqrt(
            mean_squared_error(
                y_test_eval[mascara_baseline], baseline_estacional[mascara_baseline]
            )
        )
        r2_baseline_estacional = r2_score(
            y_test_eval[mascara_baseline], baseline_estacional[mascara_baseline]
        )
    else:
        mae_baseline_estacional = np.nan
        rmse_baseline_estacional = np.nan
        r2_baseline_estacional = np.nan

    comparacion = metadata_test.copy()
    comparacion["Real"] = y_test_eval
    comparacion["Prediccion"] = predicciones
    comparacion["Baseline_Media_Train"] = baseline_media
    comparacion["Baseline_Estacional_12m"] = baseline_estacional
    comparacion["Error"] = y_test_eval - predicciones
    comparacion["Error_Absoluto"] = np.abs(comparacion["Error"])

    supera_baseline_estacional = (
        not np.isfinite(r2_baseline_estacional)
        or r2 > r2_baseline_estacional
    )

    if r2 >= 0.70 and nrmse <= 20 and supera_baseline_estacional:
        estado = "MODELO APROBADO"
    elif r2 >= 0.50 and nrmse <= 30 and supera_baseline_estacional:
        estado = "MODELO ACEPTABLE"
    else:
        estado = "MODELO REQUIERE AJUSTES"

    print()
    print("=" * 90)
    print("MÉTRICAS DE REGRESIÓN - ESCALA ORIGINAL")
    print("=" * 90)
    print()
    print(f"MAE                   : {mae:,.4f}")
    print(f"RMSE                  : {rmse:,.4f}")
    print(f"NMAE (% media |y|)    : {nmae:.2f}%")
    print(f"NRMSE (% media |y|)   : {nrmse:.2f}%")
    print("MAPE                  : " + (f"{mape:.2f}%" if np.isfinite(mape) else "No calculable"))
    print(f"R²                    : {r2:.4f}")
    print()
    print("BASELINES TEMPORALES")
    print(f"R² baseline media train     : {r2_baseline_media:.4f}")
    print(
        "R² baseline estacional 12m : "
        + (f"{r2_baseline_estacional:.4f}" if np.isfinite(r2_baseline_estacional) else "No calculable")
    )
    print(f"Modelo supera baseline 12m  : {supera_baseline_estacional}")

print()
print(f"Estado general : {estado}")
print()
display(comparacion.head(20))


## **M7. Diagnóstico, Recomendaciones y Gráficas**


In [ ]:
#==========================================================================================
# M7.1 DIAGNÓSTICO Y RECOMENDACIONES AUTOMÁTICAS
#==========================================================================================

recomendaciones = []

if TIPO_PROBLEMA.lower().startswith("clas"):

    diagnostico_df = pd.DataFrame({
        "Indicador": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "Especificidad",
            "Accuracy Balanceada",
            "AUC ROC",
            "Clases presentes en Test",
            "Estado General"
        ],
        "Valor": [
            accuracy,
            precision,
            recall,
            f1,
            especificidad,
            accuracy_balanceada,
            auc_roc,
            len(np.unique(y_test)),
            estado
        ]
    })

    proporcion_minoria_train = (
        np.min(
            np.unique(
                y_train,
                return_counts=True
            )[1]
        )
        / len(y_train)
    )

    if proporcion_minoria_train < 0.20:
        recomendaciones.append(
            "El entrenamiento está desbalanceado; comparar el experimento base "
            "con una variante usando class_weight."
        )

    if len(np.unique(y_valid)) < 2:
        recomendaciones.append(
            "Validación temporal contiene una sola clase; interpretar métricas "
            "de clasificación con cautela."
        )

    if len(np.unique(y_test)) < 2:
        recomendaciones.append(
            "Prueba temporal contiene una sola clase. Mantener este resultado "
            "como evidencia temporal y complementar con validación walk-forward."
        )

    if f1 < 0.50:
        recomendaciones.append(
            "F1 bajo: revisar balanceo, arquitectura, umbral y estabilidad temporal."
        )

else:

    diagnostico_df = pd.DataFrame({
        "Indicador": [
            "MAE", "RMSE", "NMAE_%", "NRMSE_%", "MAPE_%", "R2",
            "R2 Baseline Media Train", "R2 Baseline Estacional 12m",
            "Modelo Supera Baseline Estacional", "Transformacion Objetivo",
            "Estado General"
        ],
        "Valor": [
            mae, rmse, nmae, nrmse, mape, r2,
            r2_baseline_media, r2_baseline_estacional,
            supera_baseline_estacional, TRANSFORMACION_OBJETIVO,
            estado
        ]
    })

    if r2 < 0.50:
        recomendaciones.append(
            "R² inferior a 0.50: el modelo aún no explica suficientemente la variabilidad temporal del volumen."
        )

    if nrmse > 30:
        recomendaciones.append(
            "NRMSE superior a 30%: revisar arquitectura, predictoras y estabilidad temporal."
        )

    if np.isfinite(mape) and mape > 20:
        recomendaciones.append(
            "MAPE superior a 20%: revisar periodos y nodos con error relativo alto."
        )

    if np.isfinite(r2_baseline_estacional) and r2 <= r2_baseline_estacional:
        recomendaciones.append(
            "El LSTM no supera la persistencia estacional de 12 meses. Evaluar una variante Exp04-V2 autoregresiva incorporando rezagos históricos de VolumenUtilDiarioMasa como entradas, sin usar valores futuros."
        )

    recomendaciones.append(
        "Mantener scaler_y.pkl: fue ajustado exclusivamente con y_train y es necesario para devolver las predicciones de regresión a unidades originales en C15."
    )

recomendaciones.extend([
    "Conservar el split temporal por Nodo; no reemplazarlo por un split aleatorio.",
    "Conservar scaler.pkl junto al modelo porque fue ajustado solo con entrenamiento.",
    "Comparar variantes del modelo usando exactamente el mismo split temporal."
])

recomendaciones_df = pd.DataFrame({"Recomendacion": recomendaciones})

RUTA_DIAGNOSTICO_CSV = os.path.join(CARPETA_DIAGNOSTICOS, f"diagnostico_modelo_{ID_EJECUCION}.csv")
RUTA_DIAGNOSTICO_XLSX = os.path.join(CARPETA_DIAGNOSTICOS, f"diagnostico_modelo_{ID_EJECUCION}.xlsx")
RUTA_RECOMENDACIONES_CSV = os.path.join(CARPETA_DIAGNOSTICOS, f"recomendaciones_{ID_EJECUCION}.csv")
RUTA_COMPARACION_CSV = os.path.join(CARPETA_RESULTADOS, f"predicciones_prueba_{ID_EJECUCION}.csv")

diagnostico_df.to_csv(RUTA_DIAGNOSTICO_CSV, index=False, encoding="utf-8-sig")
diagnostico_df.to_excel(RUTA_DIAGNOSTICO_XLSX, index=False)
recomendaciones_df.to_csv(RUTA_RECOMENDACIONES_CSV, index=False, encoding="utf-8-sig")
comparacion.to_csv(RUTA_COMPARACION_CSV, index=False, encoding="utf-8-sig")

print()
print("=" * 90)
print("DIAGNÓSTICO Y RECOMENDACIONES")
print("=" * 90)
print()
display(diagnostico_df)
print()
display(recomendaciones_df)


In [ ]:
#==========================================================================================
# M7.2 CURVAS DE ENTRENAMIENTO
#==========================================================================================

RUTA_GRAFICA_LOSS = os.path.join(
    CARPETA_RESULTADOS,
    f"loss_{ID_EJECUCION}.png"
)

plt.figure(figsize=(10, 5))
plt.plot(
    historial.history["loss"],
    label="Entrenamiento",
    linewidth=2
)
plt.plot(
    historial.history["val_loss"],
    label="Validación",
    linewidth=2
)
plt.title(f"Loss - {ID_EJECUCION}")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(
    RUTA_GRAFICA_LOSS,
    dpi=300
)
plt.show()

if TIPO_PROBLEMA.lower().startswith("clas"):

    RUTA_GRAFICA_METRICA = os.path.join(
        CARPETA_RESULTADOS,
        f"accuracy_{ID_EJECUCION}.png"
    )

    plt.figure(figsize=(10, 5))
    plt.plot(
        historial.history["accuracy"],
        label="Entrenamiento",
        linewidth=2
    )
    plt.plot(
        historial.history["val_accuracy"],
        label="Validación",
        linewidth=2
    )
    plt.title(f"Accuracy - {ID_EJECUCION}")
    plt.xlabel("Época")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RUTA_GRAFICA_METRICA,
        dpi=300
    )
    plt.show()

else:

    RUTA_GRAFICA_METRICA = os.path.join(
        CARPETA_RESULTADOS,
        f"mae_{ID_EJECUCION}.png"
    )

    plt.figure(figsize=(10, 5))
    plt.plot(
        historial.history["mae"],
        label="Entrenamiento",
        linewidth=2
    )
    plt.plot(
        historial.history["val_mae"],
        label="Validación",
        linewidth=2
    )
    plt.title(f"MAE - {ID_EJECUCION}")
    plt.xlabel("Época")
    plt.ylabel("MAE")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RUTA_GRAFICA_METRICA,
        dpi=300
    )
    plt.show()

print()
print("Gráficas:")
print(RUTA_GRAFICA_LOSS)
print(RUTA_GRAFICA_METRICA)


## **M8. Exportación del Modelo y Registro del Experimento**


In [ ]:
#==========================================================================================
# M8.1 GUARDAR MODELO ENTRENADO
#==========================================================================================

NOMBRE_MODELO = (
    f"modelo_{ID_EJECUCION}_{VARIABLE_OBJETIVO}.keras"
)

RUTA_MODELO = os.path.join(
    CARPETA_MODELOS,
    NOMBRE_MODELO
)

modelo.save(
    RUTA_MODELO
)

# Copia de metadata junto al modelo para despliegue/inferencia.
RUTA_METADATA_MODELO = os.path.join(
    CARPETA_MODELOS,
    "metadata_modelo.csv"
)

metadata_tensor.to_csv(
    RUTA_METADATA_MODELO,
    index=False,
    encoding="utf-8-sig"
)

print()
print("=" * 90)
print("MODELO EXPORTADO")
print("=" * 90)
print()
print(f"Modelo : {RUTA_MODELO}")
print(f"Scaler X : {RUTA_SCALER}")
print(f"Scaler y : {RUTA_SCALER_Y}")
print(f"Metadata: {RUTA_METADATA_MODELO}")


In [ ]:
#==========================================================================================
# M8.2 REGISTRO DEL EXPERIMENTO
#==========================================================================================

FECHA_EJECUCION = datetime.now()

registro_dict = {
    "Experimento": [ID_EJECUCION],
    "Experimento_Fuente_C13": [EXPERIMENTO],
    "Fecha": [FECHA_EJECUCION],
    "Dominio": [DOMINIO],
    "Tipo_Problema": [TIPO_PROBLEMA],
    "Variable_Objetivo": [VARIABLE_OBJETIVO],
    "Variable_Objetivo_Cientifico": [VARIABLE_OBJETIVO_CIENTIFICO],
    "Modelo": [TIPO_MODELO],
    "Metodo_Transformacion": [METODO_TRANSFORMACION],
    "Scaler_Ajustado_En": ["Train"],
    "Scaler": [os.path.basename(RUTA_SCALER) if RUTA_SCALER else None],
    "Transformacion_Objetivo": [TRANSFORMACION_OBJETIVO],
    "Scaler_Objetivo": [os.path.basename(RUTA_SCALER_Y) if RUTA_SCALER_Y else None],
    "Scaler_Objetivo_Ajustado_En": ["y_train" if RUTA_SCALER_Y else "No aplica"],
    "Ventana": [VENTANA],
    "Horizonte": [HORIZONTE],
    "Variables_Predictoras": [X.shape[2]],
    "Muestras": [X.shape[0]],
    "Train": [len(X_train)],
    "Validacion": [len(X_valid)],
    "Prueba": [len(X_test)],
    "Estrategia_Split": ["Temporal por Nodo usando Fecha_Objetivo"],
    "Neuronas_LSTM": [UNIDADES_LSTM],
    "Dropout": [DROPOUT],
    "Learning_Rate": [LEARNING_RATE],
    "Batch_Size": [BATCH_SIZE],
    "Epocas_Config": [EPOCAS],
    "Epocas_Ejecutadas": [len(historial.history["loss"])],
    "Loss_Prueba": [resultado_test.get("loss")],
    "Tiempo_Segundos": [TIEMPO_ENTRENAMIENTO],
    "Estado_General": [estado]
}

if TIPO_PROBLEMA.lower().startswith("clas"):

    registro_dict.update({
        "Accuracy_Prueba": [accuracy],
        "Precision_Prueba": [precision],
        "Recall_Prueba": [recall],
        "F1_Prueba": [f1],
        "Especificidad_Prueba": [especificidad],
        "Accuracy_Balanceada_Prueba": [accuracy_balanceada],
        "AUC_ROC_Prueba": [auc_roc],
        "Clases_Test": [len(np.unique(y_test))]
    })

else:

    registro_dict.update({
        "MAE_Prueba": [mae],
        "RMSE_Prueba": [rmse],
        "NMAE_Pct_Prueba": [nmae],
        "NRMSE_Pct_Prueba": [nrmse],
        "MAPE_Pct_Prueba": [mape],
        "R2_Prueba": [r2],
        "R2_Baseline_Media_Train": [r2_baseline_media],
        "R2_Baseline_Estacional_12m": [r2_baseline_estacional],
        "Supera_Baseline_Estacional": [supera_baseline_estacional]
    })

registro_modelado = pd.DataFrame(
    registro_dict
)

RUTA_REGISTRO_CSV = os.path.join(
    CARPETA_RESULTADOS,
    f"registro_{ID_EJECUCION}.csv"
)

RUTA_REGISTRO_XLSX = os.path.join(
    CARPETA_RESULTADOS,
    f"registro_{ID_EJECUCION}.xlsx"
)

registro_modelado.to_csv(
    RUTA_REGISTRO_CSV,
    index=False,
    encoding="utf-8-sig"
)

registro_modelado.to_excel(
    RUTA_REGISTRO_XLSX,
    index=False
)

print()
print("=" * 90)
print("REGISTRO DEL EXPERIMENTO")
print("=" * 90)
print()
display(registro_modelado)


## **M9. Bitácora Maestra y Manifiesto de Artefactos**


In [ ]:
#==========================================================================================
# M9.1 BITÁCORA MAESTRA
#==========================================================================================

ARCHIVO_BITACORA_XLSX = "bitacora_maestra_experimentos.xlsx"
ARCHIVO_BITACORA_CSV = "bitacora_maestra_experimentos.csv"

if os.path.exists(
    ARCHIVO_BITACORA_XLSX
):
    bitacora = pd.read_excel(
        ARCHIVO_BITACORA_XLSX
    )
else:
    bitacora = pd.DataFrame()

bitacora = pd.concat(
    [
        bitacora,
        registro_modelado
    ],
    ignore_index=True,
    sort=False
)

bitacora.to_excel(
    ARCHIVO_BITACORA_XLSX,
    index=False
)

bitacora.to_csv(
    ARCHIVO_BITACORA_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print("=" * 90)
print("BITÁCORA MAESTRA")
print("=" * 90)
print()
display(bitacora.tail())


In [ ]:
#==========================================================================================
# M9.2 MANIFIESTO Y VALIDACIÓN FINAL DE ARTEFACTOS
#==========================================================================================

artefactos_obligatorios = [
    RUTA_TENSOR_X,
    RUTA_TENSOR_X_RAW,
    RUTA_TENSOR_Y,
    RUTA_METADATA_SPLIT,
    RUTA_METADATA_TENSOR_CSV,
    RUTA_METADATA_TENSOR_XLSX,
    RUTA_MODELO,
    RUTA_METADATA_MODELO,
    RUTA_DIAGNOSTICO_CSV,
    RUTA_DIAGNOSTICO_XLSX,
    RUTA_RECOMENDACIONES_CSV,
    RUTA_COMPARACION_CSV,
    RUTA_GRAFICA_LOSS,
    RUTA_GRAFICA_METRICA,
    RUTA_REGISTRO_CSV,
    RUTA_REGISTRO_XLSX
]

if RUTA_SCALER:
    artefactos_obligatorios.append(RUTA_SCALER)

if RUTA_SCALER_Y:
    artefactos_obligatorios.append(RUTA_SCALER_Y)

def sha256_archivo(ruta, bloque=1024 * 1024):
    hash_obj = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        while True:
            datos = archivo.read(bloque)
            if not datos:
                break
            hash_obj.update(datos)

    return hash_obj.hexdigest()

filas_manifiesto = []

for ruta in artefactos_obligatorios:

    existe = os.path.exists(
        ruta
    )

    filas_manifiesto.append({
        "Artefacto": ruta,
        "Existe": existe,
        "Bytes": (
            os.path.getsize(ruta)
            if existe
            else None
        ),
        "SHA256": (
            sha256_archivo(ruta)
            if existe
            else None
        )
    })

manifiesto = pd.DataFrame(
    filas_manifiesto
)

RUTA_MANIFIESTO = os.path.join(
    CARPETA_RESULTADOS,
    f"manifiesto_artefactos_{ID_EJECUCION}.csv"
)

manifiesto.to_csv(
    RUTA_MANIFIESTO,
    index=False,
    encoding="utf-8-sig"
)

controles_finales = {
    "Todos_artefactos_existen":
        manifiesto["Existe"].all(),

    "Modelo_existe":
        os.path.exists(RUTA_MODELO),

    "Scaler_existe_si_aplica":
        (
            True
            if RUTA_SCALER is None
            else os.path.exists(RUTA_SCALER)
        ),

    "Scaler_ajustado_solo_train":
        not TRANSFORMACION_C13,

    "Scaler_y_regresion_existe":
        (not TIPO_PROBLEMA.lower().startswith("reg")) or os.path.exists(RUTA_SCALER_Y),

    "Split_temporal_valido":
        integridad_temporal_split,

    "Sin_NaN_en_X_modelo":
        not pd.isna(X).any(),

    "Alineacion_tensores":
        len(X) == len(y),

    "Predictoras_coinciden":
        X.shape[2] == len(VARIABLES_MODELO)
}

print()
print("=" * 90)
print("VALIDACIÓN FINAL DE C14")
print("=" * 90)
print()

for control, valor in controles_finales.items():
    print(f"- {control:<38}: {valor}")

print()

if not all(controles_finales.values()):
    raise RuntimeError(
        "C14 terminó con inconsistencias en sus artefactos."
    )

print("Resultado : C14 COMPLETADO Y VALIDADO")
print()
display(manifiesto)


## **M10. Resumen Final**


In [ ]:
#==========================================================================================
# M10. RESUMEN DEL FRAMEWORK DE MODELADO
#==========================================================================================

print()
print("=" * 90)
print("FRAMEWORK V7 - C14 MODELADO FINALIZADO")
print("=" * 90)
print()
print(f"Experimento fuente C13 : {EXPERIMENTO}")
print(f"Ejecución C14          : {ID_EJECUCION}")
print(f"Dominio                : {DOMINIO}")
print(f"Tipo de problema       : {TIPO_PROBLEMA}")
print(f"Objetivo modelo        : {VARIABLE_OBJETIVO}")
print(f"Objetivo científico    : {VARIABLE_OBJETIVO_CIENTIFICO}")
print(f"Modelo                 : {TIPO_MODELO}")
print(f"Ventana / Horizonte    : {VENTANA} / {HORIZONTE}")
print(f"Predictoras            : {X.shape[2]}")
print(f"Secuencias             : {X.shape[0]}")
print(f"Train / Valid / Test   : "
      f"{len(X_train)} / {len(X_valid)} / {len(X_test)}")
print(f"Transformación         : {METODO_TRANSFORMACION}")
print("Scaler ajustado en     : C14 - SOLO entrenamiento")
print(f"Scaler X               : {RUTA_SCALER}")
print(f"Scaler y               : {RUTA_SCALER_Y}")
print(f"Modelo                 : {RUTA_MODELO}")
print(f"Estado                 : {estado}")
print()
print("C13 permanece sin escalamiento.")
print("C15 deberá cargar scaler.pkl antes de inferencia.")
if RUTA_SCALER_Y:
    print("Para regresión, C15 deberá además cargar scaler_y.pkl para desescalar la salida del modelo.")
print()
print("Proceso completado correctamente.")
